<a href="https://colab.research.google.com/github/DanielLucero25/CSC349/blob/main/HW5/Copy_of_diabetes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 349 HW5: Diabetes Classification

This notebook uses the Pima Indians Diabetes dataset to compare ensemble models. The dataset is included in the HW5 folder.


## Task 1: Load the data

I load the Pima diabetes data from a mirror into a Pandas DataFrame and give each column a name. The dataset has eight features and an outcome column. An outcome of 0 means no diabetes and 1 means diabetes. The same data file is included in the HW5 folder.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier, BaggingClassifier, RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

columns = ['pregnancies', 'glucose', 'blood_pressure', 'skin_thickness', 'insulin',
           'bmi', 'diabetes_pedigree', 'age', 'outcome']
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
df = pd.read_csv(url, names=columns)
print('Shape:', df.shape)


## Task 2: Inspect the data

I use head to look at the first five rows, info to check the data types and non-null counts, and describe to see the ranges and averages. Some medical measurements have a minimum of zero even though a zero is not a realistic measurement. These can be missing values that are not shown as blank cells.


In [ ]:
print(df.head())
print()
df.info()
print()
print(df.describe())


## Task 3: Find missing values

I count zeros in glucose, blood pressure, skin thickness, insulin, and BMI. I treat these zeros as missing and replace them with NaN. A zero for pregnancies is possible, so I leave that column alone. The SimpleImputer below uses the median to fill missing values. It is fitted later on the training data inside the pipeline so the test data does not affect the medians.


In [ ]:
missing_columns = ['glucose', 'blood_pressure', 'skin_thickness', 'insulin', 'bmi']
print('Zeros in each measurement:')
print((df[missing_columns] == 0).sum())
df[missing_columns] = df[missing_columns].replace(0, np.nan)
print('\nMissing values after replacing zeros:')
print(df.isna().sum())
imputer = SimpleImputer(strategy='median')


## Task 4: Preprocessing and train/test split

I separate the outcome from the features and split the data into training and test sets. Stratify keeps a similar share of both outcomes in each set. The pipeline fills missing values using the training medians and scales the numerical columns. ColumnTransformer applies these steps to all eight features. The scaler is helpful for logistic regression and SVC because the measurements are on different scales.


In [ ]:
X = df.drop('outcome', axis=1)
y = df['outcome']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

numeric_pipeline = Pipeline([
    ('imputer', imputer),
    ('scaler', StandardScaler())
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, X.columns.tolist())
])
print('Training shape:', X_train.shape)
print('Test shape:', X_test.shape)


## Task 5: Hard and soft voting

I combine logistic regression, an SVC, and a decision tree in two voting classifiers. Hard voting uses the majority of the models’ class predictions. Soft voting averages their predicted probabilities, so the SVC needs probability=True. Each model is placed after preprocessing in a pipeline. I compare test accuracy and classification reports, then plot the confusion matrices to see correct predictions and mistakes for each class.


In [ ]:
models = [('logistic', LogisticRegression(max_iter=1000, random_state=42)),
          ('svc', SVC(probability=True, random_state=42)),
          ('tree', DecisionTreeClassifier(random_state=42))]

hard = Pipeline([('preprocessor', preprocessor),
                 ('model', VotingClassifier(estimators=models, voting='hard'))])
soft = Pipeline([('preprocessor', preprocessor),
                 ('model', VotingClassifier(estimators=models, voting='soft'))])

hard.fit(X_train, y_train)
soft.fit(X_train, y_train)
hard_predictions = hard.predict(X_test)
soft_predictions = soft.predict(X_test)
print('Hard voting accuracy:', accuracy_score(y_test, hard_predictions))
print(classification_report(y_test, hard_predictions))
print('Soft voting accuracy:', accuracy_score(y_test, soft_predictions))
print(classification_report(y_test, soft_predictions))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(y_test, hard_predictions, ax=axes[0], colorbar=False)
axes[0].set_title('Hard voting')
ConfusionMatrixDisplay.from_predictions(y_test, soft_predictions, ax=axes[1], colorbar=False)
axes[1].set_title('Soft voting')
plt.tight_layout()
plt.show()


## Task 6: Bagging and out-of-bag score

I train a bagging classifier with decision trees. Bootstrap sampling gives each tree a random sample of the training rows with replacement. Some rows are left out for each tree and can be used for out-of-bag (OOB) evaluation. I compare the OOB score with test accuracy on one graph. Preprocessing is fitted on all training rows before bootstrapping, so the OOB score is an approximate internal check; the held-out test set provides a separate evaluation.


In [ ]:
bagging = Pipeline([
    ('preprocessor', preprocessor),
    ('model', BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42),
                                n_estimators=100, bootstrap=True, oob_score=True,
                                random_state=42))
])
bagging.fit(X_train, y_train)
oob_score = bagging.named_steps['model'].oob_score_
bagging_accuracy = bagging.score(X_test, y_test)
print('OOB score:', oob_score)
print('Test accuracy:', bagging_accuracy)

plt.bar(['OOB score', 'Test accuracy'], [oob_score, bagging_accuracy])
plt.ylim(0, 1)
plt.ylabel('Accuracy')
plt.title('Bagging results')
plt.show()


## Task 7: Random forest

I train a random forest using 100 trees. The max_features setting uses a random subset of features at each split, which helps the trees make different decisions. I check test accuracy and the classification report. The horizontal bar plot shows each feature’s mean decrease in impurity, or how much that feature helped the trees make their splits. These importances do not prove that a feature causes diabetes.


In [ ]:
forest = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, max_features='sqrt', random_state=42))
])
forest.fit(X_train, y_train)
forest_predictions = forest.predict(X_test)
print('Random forest accuracy:', accuracy_score(y_test, forest_predictions))
print(classification_report(y_test, forest_predictions))

importances = pd.Series(forest.named_steps['model'].feature_importances_, index=X.columns)
importances.sort_values().plot(kind='barh', figsize=(8, 5))
plt.xlabel('Mean decrease in impurity')
plt.title('Random forest feature importances')
plt.tight_layout()
plt.show()


## Task 8: AdaBoost

I train AdaBoost with decision stumps, which are trees with only one split. AdaBoost trains the stumps one after another and gives more attention to observations that were classified incorrectly. I compare training and test accuracy in a graph. A much higher training score could be a sign that the model is fitting the training data more closely than it generalizes to new data.


In [ ]:
ada = Pipeline([
    ('preprocessor', preprocessor),
    ('model', AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
                                 n_estimators=100, learning_rate=1.0, random_state=42))
])
ada.fit(X_train, y_train)
train_accuracy = ada.score(X_train, y_train)
test_accuracy = ada.score(X_test, y_test)
print('AdaBoost training accuracy:', train_accuracy)
print('AdaBoost test accuracy:', test_accuracy)
print(classification_report(y_test, ada.predict(X_test)))

plt.bar(['Training accuracy', 'Test accuracy'], [train_accuracy, test_accuracy])
plt.ylim(0, 1)
plt.ylabel('Accuracy')
plt.title('AdaBoost results')
plt.show()
